# **Strain For Constant Temperature with datatable**
### From week 4 exercises

#### **Exact value with Interpolation**

In [1]:
# ============================================================
# BOLTZMANN SUPERPOSITION
# CONSTANT TEMPERATURE
# SAME STRUCTURE AS N-CYCLE SCRIPT
# ============================================================

import math
import pandas as pd

# ------------------------------------------------------------
# CREEP COMPLIANCE DATA TABLE
# Units: [1/Pa]
# ------------------------------------------------------------

data = {
    "TIME [hour]": [1, 10, 100, 1000, 10000, 100000],

    23: [
        2.5e-10,
        2.6e-10,
        2.7e-10,
        3.0e-10,
        4.0e-10,
        6.5e-10
    ],

    40: [
        3.0e-10,
        3.6e-10,
        4.5e-10,
        6.3e-10,
        9.7e-10,
        1.6e-09
    ],

    60: [
        4.6e-10,
        6.4e-10,
        9.8e-10,
        1.6e-09,
        2.9e-09,
        5.5e-09
    ],
}

df = pd.DataFrame(data)

# ============================================================
# TYPE IN LOAD HISTORY HERE
# ============================================================

# Absolute stress levels [Pa]
sigma = [
    7e6,   # sigma1
    0e6,   # sigma2
]

# Duration of each load step [h]
times = [
    10,    # t1
    1,     # t2
]

# Constant temperature [°C]
temperature = 60

# ============================================================
# REFERENCE TEMPERATURE
# ============================================================

T_ref = 40

# ============================================================
# WLF SHIFT FACTOR FUNCTION
# ============================================================

def shift_factor(T, T_ref):

    delta_T = T - T_ref

    log10_aT = (
        -(8.86 * delta_T)
        /
        (101.6 + delta_T)
    )

    aT = 10 ** log10_aT

    return log10_aT, aT

# ============================================================
# CALCULATE SHIFT FACTOR
# ============================================================

print("================================================")
print("SHIFT FACTOR")
print("================================================")

log10_aT, aT = shift_factor(
    temperature,
    T_ref
)

print(f"Temperature = {temperature} °C")
print(f"Reference temperature = {T_ref} °C")
print(f"log10(aT) = {log10_aT:.6f}")
print(f"aT = {aT:.6e}")

# ============================================================
# CALCULATE TIME ARGUMENTS
# ============================================================

print("\n================================================")
print("TIME ARGUMENTS")
print("================================================")

t_arguments = []

N = len(times)

for i in range(N):

    t_argument = 0

    for j in range(i, N):

        t_argument += times[j]

    t_arguments.append(t_argument)

    print(
        f"t{i+1}_argument = "
        f"{t_argument:.6f} h"
    )

# ============================================================
# LOGARITHMIC INTERPOLATION FUNCTION
# ============================================================

def interpolate_compliance(
    t,
    temperature,
    df
):

    time_values = df["TIME [hour]"].values
    J_values = df[temperature].values

    # --------------------------------------------------------
    # EXACT MATCH
    # --------------------------------------------------------

    if any(
        abs(t - tv) < 1e-9
        for tv in time_values
    ):

        idx = min(
            range(len(time_values)),
            key=lambda i:
            abs(time_values[i] - t)
        )

        J_exact = J_values[idx]

        print(f"Temperature = {temperature} °C")

        print(
            f"Exact table value at "
            f"{t} h"
        )

        print(
            f"J = {J_exact:.3e}\n"
        )

        return J_exact

    # --------------------------------------------------------
    # FIND INTERPOLATION REGION
    # --------------------------------------------------------

    for i in range(len(time_values)-1):

        if (
            time_values[i]
            <= t
            <= time_values[i+1]
        ):

            t1 = time_values[i]
            t2 = time_values[i+1]

            J1 = J_values[i]
            J2 = J_values[i+1]

            log_t = math.log10(t)
            log_t1 = math.log10(t1)
            log_t2 = math.log10(t2)

            # ------------------------------------------------
            # LOG INTERPOLATION
            # ------------------------------------------------

            J_interp = (
                J1
                +
                (J2 - J1)
                *
                (
                    (log_t - log_t1)
                    /
                    (log_t2 - log_t1)
                )
            )

            # ------------------------------------------------
            # PRINT RESULTS
            # ------------------------------------------------

            print(f"Temperature = {temperature} °C")

            print(
                f"Interpolating between "
                f"{t1} h and {t2} h"
            )

            print(
                f"Interpolated J = "
                f"{J_interp:.3e}\n"
            )

            return J_interp

    raise ValueError(
        f"Time {t} h outside table range"
    )

# ============================================================
# CALCULATE COMPLIANCE VALUES
# ============================================================

print("\n================================================")
print("COMPLIANCE VALUES")
print("================================================")

J_values = []

for i, t_arg in enumerate(t_arguments):

    J_value = interpolate_compliance(
        t_arg,
        temperature,
        df
    )

    J_values.append(J_value)

    print(
        f"J{i+1} = "
        f"{J_value:.3e}"
    )

# ============================================================
# CALCULATE STRAIN
# ============================================================

print("\n================================================")
print("STRAIN CONTRIBUTIONS")
print("================================================")

epsilon = 0

for i in range(N):

    # First term
    if i == 0:

        delta_sigma = sigma[0]

    # Remaining terms
    else:

        delta_sigma = (
            sigma[i]
            -
            sigma[i-1]
        )

    contribution = (
        delta_sigma
        *
        J_values[i]
    )

    epsilon += contribution

    print(
        f"Contribution {i+1} = "
        f"{contribution:.6e}"
    )

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================================================")
print("FINAL RESULTS")
print("================================================")

print(
    f"Total strain = "
    f"{epsilon:.6e}"
)

print(
    f"Total strain percent = "
    f"{epsilon*100:.6f} %"
)

SHIFT FACTOR
Temperature = 60 °C
Reference temperature = 40 °C
log10(aT) = -1.457237
aT = 3.489500e-02

TIME ARGUMENTS
t1_argument = 11.000000 h
t2_argument = 1.000000 h

COMPLIANCE VALUES
Temperature = 60 °C
Interpolating between 10 h and 100 h
Interpolated J = 6.541e-10

J1 = 6.541e-10
Temperature = 60 °C
Exact table value at 1 h
J = 4.600e-10

J2 = 4.600e-10

STRAIN CONTRIBUTIONS
Contribution 1 = 4.578515e-03
Contribution 2 = -3.220000e-03

FINAL RESULTS
Total strain = 1.358515e-03
Total strain percent = 0.135851 %


In [2]:
# ============================================================
# BOLTZMANN SUPERPOSITION FOR N LOADS
# Constant temperature
# Uses creep compliance table with logarithmic interpolation
# ============================================================

import math
import pandas as pd

# ------------------------------------------------------------
# CREEP COMPLIANCE DATA TABLE
# Units: [1/Pa]
# ------------------------------------------------------------

data = {
    "TIME [hour]": [1, 10, 100, 1000, 10000, 100000],
    23: [2.5e-10, 2.6e-10, 2.7e-10, 3.0e-10, 4.0e-10, 6.5e-10],
    40: [3.0e-10, 3.6e-10, 4.5e-10, 6.3e-10, 9.7e-10, 1.6e-09],
    60: [4.6e-10, 6.4e-10, 9.8e-10, 1.6e-09, 2.9e-09, 5.5e-09],
}

df = pd.DataFrame(data)

# ============================================================
# LOGARITHMIC INTERPOLATION FUNCTION
# Returns J(t)
# ============================================================

def interpolate_compliance(t, temperature, df):

    time_values = df["TIME [hour]"].values
    J_values = df[temperature].values

    # Exact match
    if t in time_values:
        idx = list(time_values).index(t)
        return J_values[idx]

    # Find interval
    for i in range(len(time_values)-1):

        if time_values[i] <= t <= time_values[i+1]:

            t1 = time_values[i]
            t2 = time_values[i+1]

            J1 = J_values[i]
            J2 = J_values[i+1]

            # Logarithmic interpolation
            J_interp = (
                J1
                +
                (J2 - J1)
                *
                (
                    (math.log10(t) - math.log10(t1))
                    /
                    (math.log10(t2) - math.log10(t1))
                )
            )

            return J_interp

    raise ValueError("Time outside table range")

# ============================================================
# BOLTZMANN SUPERPOSITION FUNCTION
# ============================================================

def boltzmann_strain(
    sigma_changes,
    load_times,
    t_total,
    temperature,
    df
):

    epsilon = 0

    for sigma, tau in zip(sigma_changes, load_times):

        if t_total > tau:

            delta_t = t_total - tau

            J = interpolate_compliance(
                delta_t,
                temperature,
                df
            )

            epsilon += sigma * J

    return epsilon

# ============================================================
# TYPE IN LOAD HISTORY !HERE!
# ============================================================

# Stress changes [Pa]
sigma_changes = [
    7e6,     # Apply +7 MPa at t=0
    -7e6,    # Remove 7 MPa at t=10 h
]

# Times where stress changes occur [h]
load_times = [
    0,
    10,
]

# Total evaluation time [h]
t_total = 11

# Constant temperature [°C]
temperature = 60

# ============================================================
# CALCULATE STRAIN
# ============================================================

epsilon = boltzmann_strain(
    sigma_changes,
    load_times,
    t_total,
    temperature,
    df
)

# ============================================================
# RESULTS
# ============================================================

print("Strain =", epsilon)

print("Strain percent =", epsilon * 100, "%")

Strain = 0.0013585145906765757
Strain percent = 0.13585145906765758 %


#### **Approximate value without interpolation**

In [3]:
# ============================================================
# BOLTZMANN SUPERPOSITION FOR N LOADS
# WITHOUT INTERPOLATION
#
# Uses closest time match directly from the compliance table
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# CREEP COMPLIANCE DATA TABLE
# Units: [1/Pa]
# ------------------------------------------------------------

data = {
    "TIME [hour]": [1, 10, 100, 1000, 10000, 100000],
    23: [2.5e-10, 2.6e-10, 2.7e-10, 3.0e-10, 4.0e-10, 6.5e-10],
    40: [3.0e-10, 3.6e-10, 4.5e-10, 6.3e-10, 9.7e-10, 1.6e-09],
    60: [4.6e-10, 6.4e-10, 9.8e-10, 1.6e-09, 2.9e-09, 5.5e-09],
}

df = pd.DataFrame(data)

# ============================================================
# FIND CLOSEST TIME MATCH
# Returns closest J(t)
# ============================================================

def closest_compliance(t, temperature, df):

    time_values = df["TIME [hour]"].values

    # Find closest time value
    closest_time = min(
        time_values,
        key=lambda x: abs(x - t)
    )

    # Extract compliance value
    J = df.loc[
        df["TIME [hour]"] == closest_time,
        temperature
    ].iloc[0]

    print(
        f"Requested time = {t} h | "
        f"Closest table time = {closest_time} h"
    )

    return J

# ============================================================
# BOLTZMANN SUPERPOSITION FUNCTION
# ============================================================

def boltzmann_strain_closest(
    sigma_changes,
    load_times,
    t_total,
    temperature,
    df
):

    epsilon = 0

    for sigma, tau in zip(sigma_changes, load_times):

        if t_total > tau:

            delta_t = t_total - tau

            J = closest_compliance(
                delta_t,
                temperature,
                df
            )

            epsilon += sigma * J

    return epsilon

# ============================================================
# TYPE IN LOAD HISTORY !HERE!
# ============================================================

# Stress changes [Pa]
sigma_changes = [
    7e6,     # Apply +7 MPa at t=0
    -7e6,    # Remove 7 MPa at t=10 h
]

# Times where stress changes occur [h]
load_times = [
    0,
    10,
]

# Total evaluation time [h]
t_total = 11

# Constant temperature [°C]
temperature = 60

# ============================================================
# CALCULATE STRAIN
# ============================================================

epsilon = boltzmann_strain_closest(
    sigma_changes,
    load_times,
    t_total,
    temperature,
    df
)

# ============================================================
# RESULTS
# ============================================================

print("Strain =", epsilon)

print("Strain percent =", epsilon * 100, "%")

Requested time = 11 h | Closest table time = 10 h
Requested time = 1 h | Closest table time = 1 h
Strain = 0.0012599999999999994
Strain percent = 0.12599999999999995 %


# **Strain For Different Temperatures with datatable**
### From week 4 exercises

#### **Exact value with Interpolation**

In [4]:
# ============================================================
# BOLTZMANN SUPERPOSITION
# FOR N LOAD CYCLES
# USING CREEP COMPLIANCE DATA TABLE
# ============================================================

import pandas as pd
import math

# ============================================================
# INPUT MODULUS DATA TABLE
# Units: [MPa]
# ============================================================

Unit = "MPa"

data = {
    "TIME [hour]": [1, 10, 100, 1000, 10000],

    30: [880, 788, 683, 565, 464],
    50: [629, 525, 418, 356, 295],
    70: [400, 343, 283, 246, 209],
}

df_input = pd.DataFrame(data)

# ============================================================
# TYPE IN LOAD HISTORY HERE
# ============================================================

# Absolute stress levels [Pa]
sigma = [
    5e6,
    0e6,
]

# Duration of each load step [h]
times = [
    10,
    1,
]

# Temperature during each load step [°C]
temperatures = [
    70,
    30,
]

# ============================================================
# REFERENCE TEMPERATURES
# ============================================================

# WLF reference temperature
T_ref = 40

# Compliance calculation temperature
aT_ref = 30

# ============================================================
# GENERAL CONVERSION FUNCTION
# ============================================================

def modulus_to_compliance(
    df,
    Unit
):

    df_output = df.copy()

    temperature_columns = [
        col
        for col in df_output.columns
        if col != "TIME [hour]"
    ]

    # --------------------------------------------------------
    # CONVERT MPa -> 1/Pa
    # --------------------------------------------------------

    if Unit == "MPa":

        print(
            "Converting modulus [MPa] "
            "to compliance [1/Pa]\n"
        )

        for T in temperature_columns:

            E_pa = df_output[T] * 1e6

            df_output[T] = 1 / E_pa

    # --------------------------------------------------------
    # ALREADY IN COMPLIANCE FORM
    # --------------------------------------------------------

    elif Unit == "Pa":

        print(
            "Input already in compliance "
            "units [1/Pa]"
        )

        print("Skipping conversion\n")

    else:

        raise ValueError(
            'Unit must be "MPa" or "Pa"'
        )

    return df_output

# ============================================================
# CALCULATE OUTPUT TABLE
# ============================================================

df = modulus_to_compliance(
    df_input,
    Unit
)

print("\n================================================")
print("OUTPUT TABLE")
print("================================================")
print(df)

# ============================================================
# SHIFT FACTOR FUNCTION
# ============================================================

def aT_base(T, T_ref):

    delta = T - T_ref

    denom = 101.6 + delta

    if denom == 0:

        raise ZeroDivisionError(
            "101.6 + (T - T_ref) == 0"
        )

    log10_aT = (
        -(8.86 * delta)
        /
        denom
    )

    return 10 ** log10_aT

# ============================================================
# CALCULATE SHIFT FACTORS
# ============================================================

aT_values = []

print("\n================================================")
print("SHIFT FACTORS")
print("================================================")

for i, T in enumerate(temperatures):

    aT = aT_base(T, T_ref)

    aT_values.append(aT)

    print(f"aT{i+1} = {aT}")

# ============================================================
# CALCULATE EQUIVALENT TIME ARGUMENTS
# ============================================================

print("\n================================================")
print("SHIFTED TIME ARGUMENTS")
print("================================================")

t_arguments = []

N = len(times)

# ------------------------------------------------------------
# REFERENCE SHIFT FACTOR
# ------------------------------------------------------------

aT_reference = aT_base(
    aT_ref,
    T_ref
)

print(
    f"Reference calculation temperature = "
    f"{aT_ref} °C"
)

print(
    f"Reference alpha_T = "
    f"{aT_reference}"
)

# ------------------------------------------------------------
# CALCULATE TIME ARGUMENTS
# ------------------------------------------------------------

for i in range(N):

    t_argument = 0

    for j in range(i, N):

        t_argument += (

            times[j]

            *

            (

                aT_reference

                /

                aT_values[j]

            )

        )

    t_arguments.append(t_argument)

    print(
        f"t{i+1}_argument = "
        f"{t_argument}"
    )

# ============================================================
# LOGARITHMIC INTERPOLATION FUNCTION
# ============================================================

def interpolate_compliance(
    t,
    temperature,
    df
):

    time_values = df["TIME [hour]"].values

    J_values = df[temperature].values

    # --------------------------------------------------------
    # BELOW TABLE RANGE
    # --------------------------------------------------------

    if t < time_values[0]:

        print(f"Temperature = {temperature} °C")

        print(
            f"Time {t:.6f} h below table range"
        )

        print(
            f"Using minimum table value "
            f"at {time_values[0]} h"
        )

        print(
            f"J = {J_values[0]:.3e}\n"
        )

        return J_values[0]

    # --------------------------------------------------------
    # ABOVE TABLE RANGE
    # --------------------------------------------------------

    if t > time_values[-1]:

        print(f"Temperature = {temperature} °C")

        print(
            f"Time {t:.6f} h above table range"
        )

        print(
            f"Using maximum table value "
            f"at {time_values[-1]} h"
        )

        print(
            f"J = {J_values[-1]:.3e}\n"
        )

        return J_values[-1]

    # --------------------------------------------------------
    # EXACT MATCH
    # --------------------------------------------------------

    if any(
        abs(t - tv) < 1e-9
        for tv in time_values
    ):

        idx = min(
            range(len(time_values)),
            key=lambda i:
            abs(time_values[i] - t)
        )

        J_exact = J_values[idx]

        print(f"Temperature = {temperature} °C")

        print(
            f"Exact table value at "
            f"{t} h"
        )

        print(
            f"J = {J_exact:.3e}\n"
        )

        return J_exact

    # --------------------------------------------------------
    # LOG INTERPOLATION
    # --------------------------------------------------------

    for i in range(len(time_values)-1):

        if (
            time_values[i]
            <= t
            <= time_values[i+1]
        ):

            t1 = time_values[i]
            t2 = time_values[i+1]

            J1 = J_values[i]
            J2 = J_values[i+1]

            log_t = math.log10(t)
            log_t1 = math.log10(t1)
            log_t2 = math.log10(t2)

            J_interp = (

                J1

                +

                (J2 - J1)

                *

                (
                    (log_t - log_t1)
                    /
                    (log_t2 - log_t1)
                )

            )

            print(f"Temperature = {temperature} °C")

            print(
                f"Interpolating between "
                f"{t1} h and {t2} h"
            )

            print(
                f"Interpolated J = "
                f"{J_interp:.3e}\n"
            )

            return J_interp

# ============================================================
# CALCULATE COMPLIANCE VALUES
# ============================================================

print("\n================================================")
print("COMPLIANCE VALUES")
print("================================================")

J_values = []

for i, t_arg in enumerate(t_arguments):

    J_value = interpolate_compliance(
        t_arg,
        aT_ref,
        df
    )

    J_values.append(J_value)

    print(f"J{i+1} = {J_value:.3e}")

# ============================================================
# CALCULATE STRAIN
# ============================================================

print("\n================================================")
print("STRAIN CONTRIBUTIONS")
print("================================================")

epsilon = 0

for i in range(N):

    # First term
    if i == 0:

        delta_sigma = sigma[0]

    # Remaining terms
    else:

        delta_sigma = (
            sigma[i]
            -
            sigma[i-1]
        )

    contribution = (
        delta_sigma
        *
        J_values[i]
    )

    epsilon += contribution

    print(
        f"Contribution {i+1} = "
        f"{contribution}"
    )

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================================================")
print("FINAL RESULTS")
print("================================================")

print(f"Total strain = {epsilon}")

print(
    f"Total strain percent = "
    f"{epsilon * 100} %"
)


Converting modulus [MPa] to compliance [1/Pa]


OUTPUT TABLE
   TIME [hour]            30            50            70
0            1  1.136364e-09  1.589825e-09  2.500000e-09
1           10  1.269036e-09  1.904762e-09  2.915452e-09
2          100  1.464129e-09  2.392344e-09  3.533569e-09
3         1000  1.769912e-09  2.808989e-09  4.065041e-09
4        10000  2.155172e-09  3.389831e-09  4.784689e-09

SHIFT FACTORS
aT1 = 0.00955527435237959
aT2 = 9.273611719567022

SHIFTED TIME ARGUMENTS
Reference calculation temperature = 30 °C
Reference alpha_T = 9.273611719567022
t1_argument = 9706.228104996875
t2_argument = 1.0

COMPLIANCE VALUES
Temperature = 30 °C
Interpolating between 1000 h and 10000 h
Interpolated J = 2.150e-09

J1 = 2.150e-09
Temperature = 30 °C
Exact table value at 1.0 h
J = 1.136e-09

J2 = 1.136e-09

STRAIN CONTRIBUTIONS
Contribution 1 = 0.01075091737553404
Contribution 2 = -0.005681818181818182

FINAL RESULTS
Total strain = 0.005069099193715858
Total strain percent = 0.5069

# **Shift Factors and Strain with J(t) function**

In [5]:
# ============================================================
# BOLTZMANN SUPERPOSITION
# FOR N LOAD CYCLES
# DIFFERENT TEMPERATURES / TIMES / LOADS
# ============================================================

import math

# ============================================================
# TYPE IN LOAD HISTORY HERE
# ============================================================

# Absolute stress levels [Pa]
# (NOT stress changes)
sigma = [
    5e6,   # sigma1
    3e6,   # sigma2
    0e6,   # sigma3
]

# Duration of each load step [h]
times = [
    100,   # t1
    50,    # t2
    2,     # t3
]

# Temperature during each load step [°C]
temperatures = [
    50,    # T1
    20,    # T2
    40,    # T3
]
# ============================================================
# COMPLIANCE FUNCTION
# ============================================================

def J(t):

    return (
        3.688
        *
        math.exp(
            0.498 * math.log10(t)
        )
        /
        10**10
    )


# ============================================================
# REFERENCE TEMPERATURES
# ============================================================

# WLF reference temperature
T_ref = 58

# Compliance reference temperature
# Uses last temperature automatically
aT_ref = temperatures[-1]
# Or manually
#aT_ref = 40
# ============================================================
# SHIFT FACTOR FUNCTION
# ============================================================

def aT_base(T, T_ref):

    delta = T - T_ref

    denom = 101.6 + delta

    if denom == 0:

        raise ZeroDivisionError(
            "101.6 + (T - T_ref) == 0"
        )

    log10_aT = (
        -(8.86 * delta)
        /
        denom
    )

    return 10 ** log10_aT

# ============================================================
# CALCULATE ALL SHIFT FACTORS
# ============================================================

aT_values = []

print("================================================")
print("SHIFT FACTORS")
print("================================================")

for i, T in enumerate(temperatures):

    aT = aT_base(T, T_ref)

    aT_values.append(aT)

    print(f"aT{i+1} =", aT)

# ============================================================
# REFERENCE SHIFT FACTOR
# ============================================================

aT_reference = aT_values[-1]

print("\nReference alpha_T =", aT_reference)

# ============================================================
# CALCULATE SHIFTED TIME ARGUMENTS
# ============================================================

print("\n================================================")
print("SHIFTED TIME ARGUMENTS")
print("================================================")

t_arguments = []

N = len(times)

for i in range(N):

    t_argument = 0

    for j in range(i, N):

        t_argument += (
            times[j]
            *
            (
                aT_reference
                /
                aT_values[j]
            )
        )

    t_arguments.append(t_argument)

    print(
        f"t{i+1}_argument =",
        t_argument
    )

# ============================================================
# ADJUSTING TIME ARGUMENTS WITH SHIFTFACTORS
# FUNCTION VERSION
# ============================================================

def adjust_time_arguments(
    t_arguments,
    aT_reference
):

    print("\n================================================")
    print("ADJUSTED TIME ARGUMENTS WITH SHIFT FACTORS TO T_REF")
    print("================================================")

    t_arguments_adjusted = []

    for i, t_arg in enumerate(t_arguments):

        # --------------------------------------------
        # Shift time to T_ref
        # --------------------------------------------

        t_adjusted = (
            t_arg
            /
            aT_reference
        )

        t_arguments_adjusted.append(
            t_adjusted
        )

        print(
            f"t{i+1}_argument_adjusted = "
            f"{t_adjusted}"
        )

    return t_arguments_adjusted

# ============================================================
# CALCULATE ADJUSTED TIMES
# ============================================================

t_arguments_adjusted = adjust_time_arguments(
    t_arguments,
    aT_reference
)

# ============================================================
# CALCULATE COMPLIANCE VALUES
# ============================================================

print("\n================================================")
print("COMPLIANCE VALUES")
print("================================================")

J_values = []

# ------------------------------------------------------------
# USE ADJUSTED TIMES HERE
# ------------------------------------------------------------

for i, t_arg in enumerate(t_arguments_adjusted):

    J_value = J(t_arg)

    J_values.append(J_value)

    print(f"J{i+1} =", J_value)

# ============================================================
# CALCULATE STRAIN
# USING GENERAL BOLTZMANN EQUATION
# ============================================================

print("\n================================================")
print("STRAIN CONTRIBUTIONS")
print("================================================")

epsilon = 0

for i in range(N):

    # First term
    if i == 0:

        delta_sigma = sigma[0]

    # Remaining terms
    else:

        delta_sigma = (
            sigma[i]
            -
            sigma[i-1]
        )

    contribution = (
        delta_sigma
        *
        J_values[i]
    )

    epsilon += contribution

    print(
        f"Contribution {i+1} =",
        contribution
    )

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================================================")
print("FINAL RESULTS")
print("================================================")

print("Total strain =", epsilon)

print(
    "Total strain percent =",
    epsilon * 100,
    "%"
)

SHIFT FACTORS
aT1 = 5.7182739453348415
aT2 = 196657.58056251047
aT3 = 80.84543504306389

Reference alpha_T = 80.84543504306389

SHIFTED TIME ARGUMENTS
t1_argument = 1415.82891410816
t2_argument = 2.0205548738095467
t3_argument = 2.0

ADJUSTED TIME ARGUMENTS WITH SHIFT FACTORS TO T_REF
t1_argument_adjusted = 17.512787374500384
t2_argument_adjusted = 0.024992813419004535
t3_argument_adjusted = 0.024738564384428896

COMPLIANCE VALUES
J1 = 6.850182077506594e-10
J2 = 1.6606286722790633e-10
J3 = 1.6569603400126037e-10

STRAIN CONTRIBUTIONS
Contribution 1 = 0.003425091038753297
Contribution 2 = -0.00033212573445581264
Contribution 3 = -0.0004970881020037811

FINAL RESULTS
Total strain = 0.0025958772022937034
Total strain percent = 0.2595877202293703 %


In [6]:
# ============================================================
# BOLTZMANN SUPERPOSITION
# FOR N LOAD CYCLES
# DIFFERENT TEMPERATURES / TIMES / LOADS
# ============================================================

import math

# ============================================================
# TYPE IN LOAD HISTORY HERE
# ============================================================

# Absolute stress levels [Pa]
# (NOT stress changes)
sigma = [
    5e6,   # sigma1
    3e6,   # sigma2
    0e6,   # sigma3
]

# Duration of each load step [h]
times = [
    100,   # t1
    50,    # t2
    2,     # t3
]

# Temperature during each load step [°C]
temperatures = [
    50,    # T1
    20,    # T2
    40,    # T3
]
# ============================================================
# COMPLIANCE FUNCTION
# ============================================================

def J(t):

    return (
        3.688
        *
        math.exp(
            0.498 * math.log10(t)
        )
        /
        10**10
    )


# ============================================================
# REFERENCE TEMPERATURES
# ============================================================

# WLF reference temperature
T_ref = 58

# Compliance reference temperature
# Uses last temperature automatically
aT_ref = temperatures[-1]
# Or manually
#aT_ref = 40
# ============================================================
# SHIFT FACTOR FUNCTION
# ============================================================

def aT_base(T, T_ref):

    delta = T - T_ref

    denom = 101.6 + delta

    if denom == 0:

        raise ZeroDivisionError(
            "101.6 + (T - T_ref) == 0"
        )

    log10_aT = (
        -(8.86 * delta)
        /
        denom
    )

    return 10 ** log10_aT

# ============================================================
# CALCULATE ALL SHIFT FACTORS
# ============================================================

aT_values = []

print("================================================")
print("SHIFT FACTORS")
print("================================================")

for i, T in enumerate(temperatures):

    aT = aT_base(T, T_ref)

    aT_values.append(aT)

    print(f"aT{i+1} =", aT)

# ============================================================
# REFERENCE SHIFT FACTOR
# ============================================================

aT_reference = aT_values[-1]

print("\nReference alpha_T =", aT_reference)

# ============================================================
# CALCULATE SHIFTED TIME ARGUMENTS
# ============================================================

print("\n================================================")
print("SHIFTED TIME ARGUMENTS")
print("================================================")

t_arguments = []

N = len(times)

for i in range(N):

    t_argument = 0

    for j in range(i, N):

        t_argument += (
            times[j]
            *
            (
                aT_reference
                /
                aT_values[j]
            )
        )

    t_arguments.append(t_argument)

    print(
        f"t{i+1}_argument =",
        t_argument
    )

# ============================================================
# ADJUSTING TIME ARGUMENTS WITH SHIFTFACTORS
# FUNCTION VERSION
# ============================================================

def adjust_time_arguments(
    t_arguments,
    aT_reference
):

    print("\n================================================")
    print("ADJUSTED TIME ARGUMENTS WITH SHIFT FACTORS TO T_REF")
    print("================================================")

    t_arguments_adjusted = []

    for i, t_arg in enumerate(t_arguments):

        # --------------------------------------------
        # Shift time to T_ref
        # --------------------------------------------

        t_adjusted = (
            t_arg
            /
            aT_reference
        )

        t_arguments_adjusted.append(
            t_adjusted
        )

        print(
            f"t{i+1}_argument_adjusted = "
            f"{t_adjusted}"
        )

    return t_arguments_adjusted

# ============================================================
# CALCULATE ADJUSTED TIMES
# ============================================================

t_arguments_adjusted = adjust_time_arguments(
    t_arguments,
    aT_reference
)

# ============================================================
# CALCULATE COMPLIANCE VALUES
# ============================================================

print("\n================================================")
print("COMPLIANCE VALUES")
print("================================================")

J_values = []

# ------------------------------------------------------------
# USE ADJUSTED TIMES HERE
# ------------------------------------------------------------

for i, t_arg in enumerate(t_arguments_adjusted):

    J_value = J(t_arg)

    J_values.append(J_value)

    print(f"J{i+1} =", J_value)

# ============================================================
# CALCULATE STRAIN
# USING GENERAL BOLTZMANN EQUATION
# ============================================================

print("\n================================================")
print("STRAIN CONTRIBUTIONS")
print("================================================")

epsilon = 0

for i in range(N):

    # First term
    if i == 0:

        delta_sigma = sigma[0]

    # Remaining terms
    else:

        delta_sigma = (
            sigma[i]
            -
            sigma[i-1]
        )

    contribution = (
        delta_sigma
        *
        J_values[i]
    )

    epsilon += contribution

    print(
        f"Contribution {i+1} =",
        contribution
    )

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================================================")
print("FINAL RESULTS")
print("================================================")

print("Total strain =", epsilon)

print(
    "Total strain percent =",
    epsilon * 100,
    "%"
)

SHIFT FACTORS
aT1 = 5.7182739453348415
aT2 = 196657.58056251047
aT3 = 80.84543504306389

Reference alpha_T = 80.84543504306389

SHIFTED TIME ARGUMENTS
t1_argument = 1415.82891410816
t2_argument = 2.0205548738095467
t3_argument = 2.0

ADJUSTED TIME ARGUMENTS WITH SHIFT FACTORS TO T_REF
t1_argument_adjusted = 17.512787374500384
t2_argument_adjusted = 0.024992813419004535
t3_argument_adjusted = 0.024738564384428896

COMPLIANCE VALUES
J1 = 6.850182077506594e-10
J2 = 1.6606286722790633e-10
J3 = 1.6569603400126037e-10

STRAIN CONTRIBUTIONS
Contribution 1 = 0.003425091038753297
Contribution 2 = -0.00033212573445581264
Contribution 3 = -0.0004970881020037811

FINAL RESULTS
Total strain = 0.0025958772022937034
Total strain percent = 0.2595877202293703 %
